# RNNs and LSTMs — Exercises

8 exercises covering vanilla RNN, BPTT, LSTM gates, GRU, spectral stability, gradient clipping, Bahdanau attention, and character-level language modelling.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Core mechanics: RNN forward, BPTT, LSTM gates |
| ★★ | 4–6 | Deeper theory: GRU, spectral radius, gradient clipping |
| ★★★ | 7–8 | AI applications: Bahdanau attention, char-level LM |

### Topic Map

| Topic | Exercise |
|---|---|
| Vanilla RNN forward pass | 1 |
| BPTT gradient derivation | 2 |
| LSTM gate equations | 3 |
| GRU implementation | 4 |
| Spectral radius & memory | 5 |
| Gradient clipping | 6 |
| Bahdanau attention | 7 |
| Character-level LM | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f'{'PASS' if ok else 'FAIL'} — {name}')
    if not ok:
        print(f'  expected: {np.asarray(expected)}')
        print(f'  got     : {np.asarray(got)}')
    return ok

def check_true(name, cond):
    print(f'{'PASS' if cond else 'FAIL'} — {name}')
    return cond

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

print('Setup complete. Ready for exercises.')

---

## Exercise 1 ★ — Vanilla RNN Forward Pass

Implement a vanilla RNN forward pass from scratch using only NumPy.

**(a)** Implement `rnn_forward(Wh, Wx, bh, xs)` which computes hidden states $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b}_h)$ for $t = 1, \ldots, T$ with $\mathbf{h}_0 = \mathbf{0}$.

**(b)** Verify that setting $W_h = 0$ makes each hidden state depend only on the current input (no memory).

**(c)** Verify that hidden states are bounded: $\|\mathbf{h}_t\|_\infty \leq 1$ for all $t$ (tanh is bounded).

**Hint:** Use `np.tanh`, `np.concatenate`, and matrix-vector products.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 1: Your Solution
    
    def rnn_forward(Wh, Wx, bh, xs):
        """
        Vanilla RNN forward pass.
        Wh: (d, d), Wx: (d, p), bh: (d,), xs: (T, p)
        Returns: hs (T, d)
        """
        # YOUR CODE HERE
        pass
    
    # Test data
    p1, d1, T1 = 5, 8, 12
    rng1 = np.random.default_rng(1)
    Wh1 = rng1.standard_normal((d1, d1)) * 0.5
    Wx1 = rng1.standard_normal((d1, p1)) * 0.5
    bh1 = np.zeros(d1)
    xs1 = rng1.standard_normal((T1, p1))
    
    hs1 = rnn_forward(Wh1, Wx1, bh1, xs1)
    print('hs shape:', hs1)

In [ ]:
# Exercise 1: Solution

def rnn_forward(Wh, Wx, bh, xs):
    T, p = xs.shape
    d = Wh.shape[0]
    h = np.zeros(d)
    hs = []
    for t in range(T):
        h = np.tanh(Wh @ h + Wx @ xs[t] + bh)
        hs.append(h.copy())
    return np.array(hs)

p1, d1, T1 = 5, 8, 12
rng1 = np.random.default_rng(1)
Wh1 = rng1.standard_normal((d1, d1)) * 0.5
Wx1 = rng1.standard_normal((d1, p1)) * 0.5
bh1 = np.zeros(d1)
xs1 = rng1.standard_normal((T1, p1))

hs1 = rnn_forward(Wh1, Wx1, bh1, xs1)

header('Exercise 1: Vanilla RNN Forward Pass')
check_true('output shape is (T, d)', hs1.shape == (T1, d1))
check_true('all hidden states bounded by tanh: |h_t| <= 1',
           np.all(np.abs(hs1) <= 1.0 + 1e-10))

# (b) Wh = 0 means no memory
hs_nomem = rnn_forward(np.zeros((d1, d1)), Wx1, bh1, xs1)
h_step0 = np.tanh(Wx1 @ xs1[0] + bh1)
h_step5 = np.tanh(Wx1 @ xs1[5] + bh1)
check_close('Wh=0: h_0 matches memoryless MLP', hs_nomem[0], h_step0)
check_close('Wh=0: h_5 matches memoryless MLP', hs_nomem[5], h_step5)

# (c) h depends on history when Wh != 0
check_true('Wh!=0: h_5 differs from memoryless',
           not np.allclose(hs1[5], h_step5))

print(f'\nFirst hidden state: {hs1[0][:4].round(3)}')
print(f'Last  hidden state: {hs1[-1][:4].round(3)}')
print('\nTakeaway: tanh bounds hidden states to (-1,1); setting Wh=0 removes memory.')

---

## Exercise 2 ★ — BPTT Gradient Verification

Derive and verify the BPTT gradient numerically.

**(a)** Implement `rnn_loss(Wh, Wx, bh, xs, targets)`: a scalar MSE loss over all steps (predict first hidden unit, target is `targets[t]`).

**(b)** Implement `numerical_grad_Wh(Wh, Wx, bh, xs, targets, eps=1e-5)`: compute $\frac{\partial \mathcal{L}}{\partial W_h}$ via finite differences.

**(c)** Implement `bptt_grad_Wh(Wh, Wx, bh, xs, targets)`: compute the same gradient analytically via BPTT (manual chain rule).

**(d)** Verify that analytic matches numerical to within $10^{-5}$.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 2: Your Solution
    
    def rnn_loss(Wh, Wx, bh, xs, targets):
        """MSE loss: sum of (h_t[0] - targets[t])^2 / 2."""
        # YOUR CODE HERE
        pass
    
    def numerical_grad_Wh(Wh, Wx, bh, xs, targets, eps=1e-5):
        """Finite-difference gradient w.r.t. Wh."""
        # YOUR CODE HERE
        pass
    
    def bptt_grad_Wh(Wh, Wx, bh, xs, targets):
        """Analytic BPTT gradient w.r.t. Wh."""
        # YOUR CODE HERE
        pass
    
    # Test data
    rng2 = np.random.default_rng(2)
    p2, d2, T2 = 3, 4, 6
    Wh2 = rng2.standard_normal((d2, d2)) * 0.3
    Wx2 = rng2.standard_normal((d2, p2)) * 0.3
    bh2 = np.zeros(d2)
    xs2 = rng2.standard_normal((T2, p2))
    tgt2 = rng2.standard_normal(T2) * 0.1
    
    print(rnn_loss(Wh2, Wx2, bh2, xs2, tgt2))
    print(numerical_grad_Wh(Wh2, Wx2, bh2, xs2, tgt2))
    print(bptt_grad_Wh(Wh2, Wx2, bh2, xs2, tgt2))

In [ ]:
# Exercise 2: Solution

def rnn_loss(Wh, Wx, bh, xs, targets):
    T, _ = xs.shape; d = Wh.shape[0]
    h = np.zeros(d); loss = 0.0
    for t in range(T):
        h = np.tanh(Wh @ h + Wx @ xs[t] + bh)
        loss += 0.5 * (h[0] - targets[t]) ** 2
    return loss

def numerical_grad_Wh(Wh, Wx, bh, xs, targets, eps=1e-5):
    grad = np.zeros_like(Wh)
    for i in range(Wh.shape[0]):
        for j in range(Wh.shape[1]):
            Wp = Wh.copy(); Wp[i,j] += eps
            Wm = Wh.copy(); Wm[i,j] -= eps
            grad[i,j] = (rnn_loss(Wp, Wx, bh, xs, targets) -
                         rnn_loss(Wm, Wx, bh, xs, targets)) / (2 * eps)
    return grad

def bptt_grad_Wh(Wh, Wx, bh, xs, targets):
    T, _ = xs.shape; d = Wh.shape[0]
    # Forward
    hs = [np.zeros(d)]
    for t in range(T):
        h = np.tanh(Wh @ hs[-1] + Wx @ xs[t] + bh)
        hs.append(h)
    # Backward
    dh = np.zeros(d)
    dWh = np.zeros_like(Wh)
    for t in reversed(range(T)):
        dloss_dh0 = np.zeros(d)
        dloss_dh0[0] = hs[t+1][0] - targets[t]
        dh = dh + dloss_dh0
        dtanh = 1 - hs[t+1] ** 2
        dz = dtanh * dh
        dWh += np.outer(dz, hs[t])
        dh = Wh.T @ dz
    return dWh

rng2 = np.random.default_rng(2)
p2, d2, T2 = 3, 4, 6
Wh2 = rng2.standard_normal((d2, d2)) * 0.3
Wx2 = rng2.standard_normal((d2, p2)) * 0.3
bh2 = np.zeros(d2)
xs2 = rng2.standard_normal((T2, p2))
tgt2 = rng2.standard_normal(T2) * 0.1

g_num = numerical_grad_Wh(Wh2, Wx2, bh2, xs2, tgt2)
g_bptt = bptt_grad_Wh(Wh2, Wx2, bh2, xs2, tgt2)

header('Exercise 2: BPTT Gradient Verification')
max_err = np.max(np.abs(g_num - g_bptt))
print(f'Max absolute error: {max_err:.2e}')
check_true('Analytic matches numerical to 1e-5', max_err < 1e-5)
print(f'\nSample gradient row: {g_bptt[0].round(6)}')
print('\nTakeaway: BPTT is backpropagation through the unrolled computation graph; '
      'the gradient decomposes as a sum over time steps.')

---

## Exercise 3 ★★ — LSTM Gate Equations and Cell State Highway

Implement all four LSTM gate equations and verify the cell-state gradient highway.

**(a)** Implement `lstm_step(h_prev, c_prev, x, W, b)` computing $\mathbf{f}_t, \mathbf{i}_t, \tilde{\mathbf{c}}_t, \mathbf{o}_t, \mathbf{c}_t, \mathbf{h}_t$.

**(b)** Implement `lstm_forward(xs, W, b)` running the LSTM over a sequence.

**(c)** Set all forget gates to 1 ($\mathbf{f}_t = \mathbf{1}$) and show that $\|\frac{\partial \mathbf{c}_T}{\partial \mathbf{c}_0}\|_F \approx \sqrt{d}$ (near-identity Jacobian).

**(d)** Set all forget gates to 0 and show the gradient vanishes.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 3: Your Solution
    
    def lstm_step(h_prev, c_prev, x, W, b):
        """
        W: (4d, d+p)  stacked gate weights [W_f; W_i; W_c; W_o]
        b: (4d,)       stacked biases
        Returns: h, c, (f, i, c_tilde, o)
        """
        # YOUR CODE HERE
        pass
    
    def lstm_forward(xs, W, b):
        """xs: (T, p), returns hs (T,d), cs (T,d)"""
        # YOUR CODE HERE
        pass
    
    # Test
    p3, d3, T3 = 6, 10, 8
    rng3 = np.random.default_rng(3)
    W3 = rng3.standard_normal((4*d3, d3+p3)) * 0.1
    b3 = np.zeros(4*d3); b3[:d3] = 1.0  # forget gate bias = 1
    xs3 = rng3.standard_normal((T3, p3))
    print(lstm_forward(xs3, W3, b3))

In [ ]:
# Exercise 3: Solution

def lstm_step(h_prev, c_prev, x, W, b):
    d = len(h_prev)
    inp = np.concatenate([h_prev, x])
    gates = W @ inp + b
    f = sigmoid(gates[0*d:1*d])
    i = sigmoid(gates[1*d:2*d])
    c_tilde = np.tanh(gates[2*d:3*d])
    o = sigmoid(gates[3*d:4*d])
    c = f * c_prev + i * c_tilde
    h = o * np.tanh(c)
    return h, c, (f, i, c_tilde, o)

def lstm_forward(xs, W, b):
    T, p = xs.shape
    d = W.shape[0] // 4
    h, c = np.zeros(d), np.zeros(d)
    hs, cs = [], []
    for t in range(T):
        h, c, _ = lstm_step(h, c, xs[t], W, b)
        hs.append(h.copy()); cs.append(c.copy())
    return np.array(hs), np.array(cs)

p3, d3, T3 = 6, 10, 8
rng3 = np.random.default_rng(3)
W3 = rng3.standard_normal((4*d3, d3+p3)) * 0.1
b3 = np.zeros(4*d3); b3[:d3] = 1.0
xs3 = rng3.standard_normal((T3, p3))
hs3, cs3 = lstm_forward(xs3, W3, b3)

header('Exercise 3: LSTM Gate Equations')
check_true('hs shape correct', hs3.shape == (T3, d3))
check_true('cs shape correct', cs3.shape == (T3, d3))
check_true('hidden states bounded by tanh', np.all(np.abs(hs3) <= 1.0 + 1e-10))

# (c) Gradient through cell state when f=1
# d c_T / d c_0 = product of diag(f_t) = I when all f_t = 1
T_grad = 20
# Simulate: product of forget gate diagonals
grad_f1 = np.ones(d3)  # diag entries
for _ in range(T_grad):
    grad_f1 *= 1.0  # f = 1, so prod stays 1
print(f'\n(c) Forget=1: ||grad||_F = {la.norm(grad_f1):.4f}  (expected {np.sqrt(d3):.4f} = sqrt(d))')
check_close('f=1: ||dc_T/dc_0||_F = sqrt(d)', la.norm(grad_f1), np.sqrt(d3))

# (d) Gradient when f=0
grad_f0 = np.ones(d3)
for _ in range(T_grad):
    grad_f0 *= 0.0  # f = 0
print(f'(d) Forget=0: ||grad||_F = {la.norm(grad_f0):.4e}  (vanished)')
check_true('f=0: gradient vanishes', la.norm(grad_f0) < 1e-10)
print('\nTakeaway: LSTM cell state is a gradient highway; forget gate controls gradient flow.')

---

## Exercise 4 ★★ — GRU Implementation

Implement the GRU and compare with the LSTM.

**(a)** Implement `gru_step(h_prev, x, Wz, Wr, Wh, bz, br, bh)` computing update gate $\mathbf{z}_t$, reset gate $\mathbf{r}_t$, candidate $\tilde{\mathbf{h}}_t$, and new hidden state $\mathbf{h}_t$.

**(b)** Implement `gru_forward(xs, Wz, Wr, Wh_g, bz, br, bh_g)`.

**(c)** Count parameters for GRU vs LSTM with $d=128$, $p=64$. Verify GRU has 25% fewer.

**(d)** Verify: $\mathbf{z}_t \approx 0$ copies previous state; $\mathbf{z}_t \approx 1$ replaces it.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 4: Your Solution
    
    def gru_step(h_prev, x, Wz, Wr, Wh_g, bz, br, bh_g):
        """
        GRU step.
        Returns: h_new, (z, r, h_cand)
        """
        # YOUR CODE HERE
        pass
    
    def gru_forward(xs, Wz, Wr, Wh_g, bz, br, bh_g):
        """Run GRU over sequence xs: (T, p). Returns hs: (T, d)."""
        # YOUR CODE HERE
        pass
    
    p4, d4, T4 = 6, 10, 8
    rng4 = np.random.default_rng(4)
    n4 = d4 + p4
    Wz4 = rng4.standard_normal((d4, n4)) * 0.1
    Wr4 = rng4.standard_normal((d4, n4)) * 0.1
    Wh4 = rng4.standard_normal((d4, n4)) * 0.1
    bz4 = np.zeros(d4); br4 = np.zeros(d4); bh4 = np.zeros(d4)
    xs4 = rng4.standard_normal((T4, p4))
    print(gru_forward(xs4, Wz4, Wr4, Wh4, bz4, br4, bh4))

In [ ]:
# Exercise 4: Solution

def gru_step(h_prev, x, Wz, Wr, Wh_g, bz, br, bh_g):
    inp = np.concatenate([h_prev, x])
    z = sigmoid(Wz @ inp + bz)
    r = sigmoid(Wr @ inp + br)
    h_cand = np.tanh(Wh_g @ np.concatenate([r * h_prev, x]) + bh_g)
    h = (1 - z) * h_prev + z * h_cand
    return h, (z, r, h_cand)

def gru_forward(xs, Wz, Wr, Wh_g, bz, br, bh_g):
    T, p = xs.shape; d = Wz.shape[0]
    h = np.zeros(d); hs = []
    for t in range(T):
        h, _ = gru_step(h, xs[t], Wz, Wr, Wh_g, bz, br, bh_g)
        hs.append(h.copy())
    return np.array(hs)

p4, d4, T4 = 6, 10, 8
rng4 = np.random.default_rng(4)
n4 = d4 + p4
Wz4 = rng4.standard_normal((d4, n4)) * 0.1
Wr4 = rng4.standard_normal((d4, n4)) * 0.1
Wh4 = rng4.standard_normal((d4, n4)) * 0.1
bz4 = np.zeros(d4); br4 = np.zeros(d4); bh4 = np.zeros(d4)
xs4 = rng4.standard_normal((T4, p4))

hs4 = gru_forward(xs4, Wz4, Wr4, Wh4, bz4, br4, bh4)

header('Exercise 4: GRU vs LSTM')
check_true('GRU output shape', hs4.shape == (T4, d4))

# (c) Parameter count
d_big, p_big, k_big = 128, 64, 1000
lstm_cell_params = 4 * d_big * (d_big + p_big) + 4 * d_big
gru_cell_params  = 3 * d_big * (d_big + p_big) + 3 * d_big
output_head_params = k_big * d_big + k_big
lstm_params = lstm_cell_params + output_head_params
gru_params  = gru_cell_params + output_head_params
print(f'\n(c) d={d_big}, p={p_big}, k={k_big}:')
print(f'  LSTM recurrent-cell params: {lstm_cell_params:,}')
print(f'  GRU  recurrent-cell params: {gru_cell_params:,}')
print(f'  Shared output-head params:  {output_head_params:,}')
print(f'  LSTM total params:          {lstm_params:,}')
print(f'  GRU  total params:          {gru_params:,}')
cell_ratio = gru_cell_params / lstm_cell_params
total_ratio = gru_params / lstm_params
print(f'  GRU/LSTM recurrent-cell ratio: {cell_ratio:.3f}  (expected ~0.75)')
print(f'  GRU/LSTM total-model ratio:    {total_ratio:.3f}')
check_true('GRU cell has ~25% fewer params than LSTM cell', 0.70 < cell_ratio < 0.80)

# (d) z=0 copies, z=1 replaces
h_p = np.array([1.0, -1.0, 0.5])
x_t = np.zeros(3)
# Force z=0: large negative bias on Wz
Wz_z0 = np.zeros((3, 6)); bz_z0 = np.full(3, -100.0)
Wr_t = np.zeros((3, 6)); Wh_t = np.zeros((3, 6))
brt = np.zeros(3); bht = np.zeros(3)
h_out_z0, (z0, _, _) = gru_step(h_p, x_t, Wz_z0, Wr_t, Wh_t, bz_z0, brt, bht)
check_close('z~0: copies h_prev', h_out_z0, h_p, tol=0.01)
print('\nTakeaway: The GRU gate interpolates between copying and updating, and the recurrent cell itself uses about 25% fewer parameters than an LSTM cell.')


---

## Exercise 5 ★★ — Spectral Radius and Memory Horizon

Analyse how spectral radius controls gradient norm and effective memory.

**(a)** Implement `make_scaled_orthogonal(d, rho)`: return a $d \times d$ matrix with spectral radius exactly `rho`.

**(b)** Implement `jacobian_product_norm(Wh, T)`: compute $\|W_h^T\|_F$.

**(c)** For $\rho \in \{0.5, 0.9, 1.0, 1.05\}$ and $T \in \{1, \ldots, 80\}$, plot $\|W_h^T\|_F$ vs $T$ on a log scale.

**(d)** Implement `memory_horizon(Wh, threshold=0.01)`: the largest $k$ such that $\|W_h^k\|_F > \text{threshold}$. Verify that horizon $\approx 1/(1-\rho)$ for $\rho < 1$.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 5: Your Solution
    
    def make_scaled_orthogonal(d, rho, seed=0):
        """Return d x d matrix with spectral radius = rho."""
        # YOUR CODE HERE
        pass
    
    def jacobian_product_norm(Wh, T):
        """Compute ||Wh^T||_F."""
        # YOUR CODE HERE
        pass
    
    def memory_horizon(Wh, threshold=0.01):
        """Largest k such that ||Wh^k||_F > threshold."""
        # YOUR CODE HERE
        pass
    
    print(make_scaled_orthogonal(5, 0.9))
    print(jacobian_product_norm(np.eye(5), 10))
    print(memory_horizon(0.9 * np.eye(5)))

In [ ]:
# Exercise 5: Solution

def make_scaled_orthogonal(d, rho, seed=0):
    rng = np.random.default_rng(seed)
    Q, _ = la.qr(rng.standard_normal((d, d)))
    return rho * Q

def jacobian_product_norm(Wh, T):
    J = np.eye(Wh.shape[0])
    for _ in range(T):
        J = Wh @ J
    return la.norm(J, 'fro')

def memory_horizon(Wh, threshold=0.01):
    J = np.eye(Wh.shape[0])
    for k in range(1, 300):
        J = Wh @ J
        if la.norm(J, 'fro') < threshold:
            return k - 1
    return 300

d5 = 8
radii = [0.5, 0.9, 1.0, 1.05]

header('Exercise 5: Spectral Radius and Memory Horizon')

# (c) Plot
T_arr = range(1, 81)
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
colors5 = ['#d62728', '#ff7f0e', '#2ca02c', '#9467bd']
for rho, col in zip(radii, colors5):
    Wh5 = make_scaled_orthogonal(d5, rho)
    norms5 = [jacobian_product_norm(Wh5, t) for t in T_arr]
    if HAS_MPL:
        ax.semilogy(list(T_arr), [max(n, 1e-15) for n in norms5],
                    label=f'rho={rho}', color=col)

if HAS_MPL:
    ax.axhline(1.0, color='k', linestyle='--', alpha=0.4)
    ax.set_xlabel('T'); ax.set_ylabel('||Wh^T||_F (log)')
    ax.set_title('Gradient Norm vs Steps Back')
    ax.legend(); plt.tight_layout(); plt.show()

# (d) Memory horizon
import math
print(f'\n{"rho":>6} {"horizon (measured)":>22} {"1/(1-rho) (analytic)":>22}')
print('-' * 55)
for rho in [0.5, 0.7, 0.9, 0.95]:
    Wh5 = make_scaled_orthogonal(d5, rho)
    h_measured = memory_horizon(Wh5)
    h_analytic = int(round(1 / (1 - rho)))
    print(f'{rho:>6.2f} {h_measured:>22} {h_analytic:>22}')
print('\nTakeaway: Memory horizon scales as 1/(1-rho); edge of chaos rho~1 maximises memory.')

---

## Exercise 6 ★★ — Gradient Clipping

Implement gradient clipping and demonstrate that it preserves gradient direction.

**(a)** Implement `clip_grad_norm(grads, threshold)`: given a list of gradient arrays, rescale all of them by $\tau / \|\mathbf{g}\|$ if $\|\mathbf{g}\| > \tau$.

**(b)** Verify: after clipping, $\|\mathbf{g}_{\text{clipped}}\| \leq \tau$.

**(c)** Verify: the direction is preserved — $\cos(\mathbf{g}_{\text{clipped}}, \mathbf{g}_{\text{original}}) = 1$.

**(d)** Show that clipping is inactive when $\|\mathbf{g}\| \leq \tau$.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 6: Your Solution
    
    def clip_grad_norm(grads, threshold):
        """
        Global gradient norm clipping.
        grads: list of np.ndarray
        Returns: (clipped_grads, global_norm_before_clip)
        """
        # YOUR CODE HERE
        pass
    
    # Test: large gradient
    rng6 = np.random.default_rng(6)
    g1_6 = rng6.standard_normal((20, 20)) * 50
    g2_6 = rng6.standard_normal(20) * 50
    tau6 = 5.0
    clipped, norm_before = clip_grad_norm([g1_6, g2_6], tau6)
    print('norm before:', norm_before)
    print('clipped:', clipped)

In [ ]:
# Exercise 6: Solution

def clip_grad_norm(grads, threshold):
    total_norm = np.sqrt(sum(la.norm(g) ** 2 for g in grads))
    if total_norm > threshold:
        factor = threshold / total_norm
        grads = [g * factor for g in grads]
    return grads, total_norm

rng6 = np.random.default_rng(6)
g1_6 = rng6.standard_normal((20, 20)) * 50
g2_6 = rng6.standard_normal(20) * 50
tau6 = 5.0

[g1c, g2c], norm_before = clip_grad_norm([g1_6, g2_6], tau6)
norm_after = np.sqrt(la.norm(g1c)**2 + la.norm(g2c)**2)

header('Exercise 6: Gradient Clipping')
print(f'Global norm before clipping: {norm_before:.2f}')
print(f'Global norm after  clipping: {norm_after:.6f}  (threshold: {tau6})')
check_true('Norm after clipping <= threshold', norm_after <= tau6 + 1e-9)

# (c) Direction preserved
cos_sim = (np.dot(g1_6.ravel(), g1c.ravel()) /
           (la.norm(g1_6) * la.norm(g1c)))
check_close('Cosine similarity = 1 (direction preserved)', cos_sim, 1.0)

# (d) Inactive when norm < tau
small_g = [np.ones((2, 2))]  # norm = 2 < tau=5
[small_gc], sn = clip_grad_norm(small_g, tau6)
check_true('Clipping inactive when norm < tau', np.allclose(small_g[0], small_gc))
print(f'  small gradient norm={sn:.2f} < tau={tau6}: unchanged')
print('\nTakeaway: Gradient clipping caps magnitude but preserves direction; '
      'used in every major LLM training run (GPT-3, LLaMA, Gemini).')

---

## Exercise 7 ★★★ — Bahdanau Attention

Implement Bahdanau (additive) attention over an LSTM encoder-decoder.

**(a)** Implement `attention_score(enc_states, dec_h, Wa, Ua, v)` computing $e_{t,s} = \mathbf{v}^\top \tanh(W_a \mathbf{h}_s^{(\text{enc})} + U_a \mathbf{h}_{t-1}^{(\text{dec})})$.

**(b)** Implement `attention_weights(enc_states, dec_h, Wa, Ua, v)` computing $\alpha_{t,s} = \text{softmax}_s(e_{t,s})$.

**(c)** Implement `context_vector(enc_states, alpha)` computing $\mathbf{c}_t = \sum_s \alpha_{t,s} \mathbf{h}_s^{(\text{enc})}$.

**(d)** Run the full attention mechanism over 5 decoder steps on a source sequence of length 8. Visualise the attention weight matrix and verify each row sums to 1.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 7: Your Solution
    
    def attention_score(enc_states, dec_h, Wa, Ua, v):
        """
        enc_states: (T_x, enc_dim)
        dec_h: (dec_dim,)
        Returns: e (T_x,)
        """
        # YOUR CODE HERE
        pass
    
    def attention_weights(enc_states, dec_h, Wa, Ua, v):
        """Returns alpha (T_x,) — softmax of scores."""
        # YOUR CODE HERE
        pass
    
    def context_vector(enc_states, alpha):
        """Returns c (enc_dim,)"""
        # YOUR CODE HERE
        pass
    
    T_x7, T_y7, enc_d7, dec_d7, attn_d7 = 8, 5, 12, 12, 6
    rng7 = np.random.default_rng(7)
    enc_states7 = rng7.standard_normal((T_x7, enc_d7))
    Wa7 = rng7.standard_normal((attn_d7, enc_d7)) * 0.1
    Ua7 = rng7.standard_normal((attn_d7, dec_d7)) * 0.1
    v7  = rng7.standard_normal(attn_d7) * 0.1
    dec_h7 = np.zeros(dec_d7)
    print(attention_weights(enc_states7, dec_h7, Wa7, Ua7, v7))

In [ ]:
# Exercise 7: Solution

def attention_score(enc_states, dec_h, Wa, Ua, v):
    enc_proj = enc_states @ Wa.T  # (T_x, attn_dim)
    dec_proj = dec_h @ Ua.T       # (attn_dim,)
    e = np.tanh(enc_proj + dec_proj) @ v
    return e

def attention_weights(enc_states, dec_h, Wa, Ua, v):
    e = attention_score(enc_states, dec_h, Wa, Ua, v)
    e = e - e.max()  # numerical stability
    alpha = np.exp(e); alpha /= alpha.sum()
    return alpha

def context_vector(enc_states, alpha):
    return alpha @ enc_states

T_x7, T_y7, enc_d7, dec_d7, attn_d7 = 8, 5, 12, 12, 6
rng7 = np.random.default_rng(7)
enc_states7 = rng7.standard_normal((T_x7, enc_d7))
Wa7 = rng7.standard_normal((attn_d7, enc_d7)) * 0.1
Ua7 = rng7.standard_normal((attn_d7, dec_d7)) * 0.1
v7  = rng7.standard_normal(attn_d7) * 0.1

alpha_matrix7 = np.zeros((T_y7, T_x7))
dec_h7 = np.zeros(dec_d7)
for t in range(T_y7):
    alpha7 = attention_weights(enc_states7, dec_h7, Wa7, Ua7, v7)
    ctx7 = context_vector(enc_states7, alpha7)
    alpha_matrix7[t] = alpha7
    dec_h7 = np.tanh(ctx7[:dec_d7] + dec_h7)

header('Exercise 7: Bahdanau Attention')
check_close('Each attention row sums to 1', alpha_matrix7.sum(axis=1), np.ones(T_y7))
check_true('Alpha in [0,1]', np.all(alpha_matrix7 >= 0) and np.all(alpha_matrix7 <= 1))
check_true('Context vector shape', ctx7.shape == (enc_d7,))

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(alpha_matrix7, cmap='viridis', aspect='auto',
                   vmin=0, vmax=alpha_matrix7.max())
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('Encoder position (source)'); ax.set_ylabel('Decoder step')
    ax.set_title('Attention Weights alpha_{t,s}')
    plt.tight_layout(); plt.show()

print('\nTakeaway: Bahdanau attention lets decoder query all encoder states — '
      'direct precursor to transformer self-attention.')

---

## Exercise 8 ★★★ — Character-Level Language Model: RNN vs LSTM

Implement and compare a character-level RNN and LSTM language model.

**(a)** Implement `char_lm_loss(model_type, params, text_ids, V)`: compute cross-entropy loss over a sequence of character IDs using either RNN or LSTM hidden state dynamics.

**(b)** Implement `compute_perplexity_lm(model_type, params, text_ids, V)`: $\text{PP} = \exp(\mathcal{L}/T)$.

**(c)** Initialise both RNN and LSTM with $d=32$, $V=27$ (26 letters + space). Compare their initial perplexity (before any training).

**(d)** Implement one step of gradient descent on the RNN and show loss decreases.

**(e)** Explain (in a print statement) why the trained LSTM would outperform the RNN on long sequences.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 8: Your Solution
    
    def char_lm_step_rnn(h, x_onehot, Wh, Wx, bh, Wo, bo):
        """One RNN step: returns (h_new, log_probs)."""
        # YOUR CODE HERE
        pass
    
    def compute_perplexity_lm(text_ids, V, d, model='rnn', seed=0):
        """Compute perplexity of an untrained RNN or LSTM LM."""
        # YOUR CODE HERE
        pass
    
    vocab = 'abcdefghijklmnopqrstuvwxyz '
    V8 = len(vocab)
    char2id8 = {c: i for i, c in enumerate(vocab)}
    text8 = 'the quick brown fox jumps over the lazy dog'
    ids8 = [char2id8[c] for c in text8 if c in char2id8]
    print('RNN perplexity:', compute_perplexity_lm(ids8, V8, 32, 'rnn'))
    print('LSTM perplexity:', compute_perplexity_lm(ids8, V8, 32, 'lstm'))

In [ ]:
# Exercise 8: Solution

def char_lm_loss_rnn(text_ids, V, d, seed=0):
    """Cross-entropy loss for an untrained RNN LM."""
    rng = np.random.default_rng(seed)
    sc = 0.1
    Wh = rng.standard_normal((d, d)) * sc
    Wx = rng.standard_normal((d, V)) * sc
    bh = np.zeros(d)
    Wo = rng.standard_normal((V, d)) * sc
    bo = np.zeros(V)
    h = np.zeros(d); total_loss = 0.0; T = len(text_ids) - 1
    for t in range(T):
        x = np.zeros(V); x[text_ids[t]] = 1.0
        h = np.tanh(Wh @ h + Wx @ x + bh)
        z = Wo @ h + bo
        z -= z.max()
        log_probs = z - np.log(np.exp(z).sum())
        total_loss -= log_probs[text_ids[t+1]]
    return total_loss / T

def char_lm_loss_lstm(text_ids, V, d, seed=0):
    """Cross-entropy loss for an untrained LSTM LM."""
    rng = np.random.default_rng(seed)
    sc = 0.1
    W_lstm = rng.standard_normal((4*d, d+V)) * sc
    b_lstm = np.zeros(4*d); b_lstm[:d] = 1.0  # forget bias
    Wo = rng.standard_normal((V, d)) * sc
    bo = np.zeros(V)
    h, c = np.zeros(d), np.zeros(d)
    total_loss = 0.0; T = len(text_ids) - 1
    for t in range(T):
        x = np.zeros(V); x[text_ids[t]] = 1.0
        inp = np.concatenate([h, x])
        gates = W_lstm @ inp + b_lstm
        f = sigmoid(gates[0*d:1*d])
        i = sigmoid(gates[1*d:2*d])
        c_tilde = np.tanh(gates[2*d:3*d])
        o = sigmoid(gates[3*d:4*d])
        c = f * c + i * c_tilde
        h = o * np.tanh(c)
        z = Wo @ h + bo; z -= z.max()
        log_probs = z - np.log(np.exp(z).sum())
        total_loss -= log_probs[text_ids[t+1]]
    return total_loss / T

vocab8 = 'abcdefghijklmnopqrstuvwxyz '
V8 = len(vocab8)
char2id8 = {c: i for i, c in enumerate(vocab8)}
text8 = 'the quick brown fox jumps over the lazy dog'
ids8 = [char2id8[c] for c in text8 if c in char2id8]
d8 = 32

rnn_ce = char_lm_loss_rnn(ids8, V8, d8)
lstm_ce = char_lm_loss_lstm(ids8, V8, d8)

header('Exercise 8: Character-Level LM Perplexity')
print(f'Untrained RNN  CE loss: {rnn_ce:.4f}  PPL: {np.exp(rnn_ce):.2f}')
print(f'Untrained LSTM CE loss: {lstm_ce:.4f}  PPL: {np.exp(lstm_ce):.2f}')
print(f'Random baseline PPL: {V8} = {V8} (uniform over vocabulary)')
check_true('RNN perplexity near V (random init)', 0.5 * V8 < np.exp(rnn_ce) < 3 * V8)
check_true('LSTM perplexity near V (random init)', 0.5 * V8 < np.exp(lstm_ce) < 3 * V8)

print('\n(e) Why LSTM outperforms RNN on long sequences:')
print('  - LSTM cell state gradient: diag(f_t) — no W_h multiplied through time')
print('  - RNN gradient: product of D_t * W_h — vanishes if rho(W_h) < 1')
print('  - Long text needs to remember characters seen 20-50 steps ago')
print('  - Empirically: LSTM PPL ~3-5 chars, RNN PPL ~8-15 chars after training')
print('\nTakeaway: LSTM cell state highway allows gradients (and information) to '
      'flow freely across many time steps; this is the core architectural advantage.')

---

## What to Review After Finishing

- [ ] Vanilla RNN recurrence: $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$
- [ ] BPTT gradient = product of Jacobians; vanishes when $\rho(W_h) < 1$
- [ ] LSTM four gates: forget, input, candidate, output; additive cell state update
- [ ] GRU: two gates, one state, 25% fewer params than LSTM
- [ ] Spectral radius: memory horizon $\approx 1/(1-\rho)$; orthogonal init gives $\rho = 1$
- [ ] Gradient clipping: preserves direction, caps magnitude; standard in all LLM training
- [ ] Bahdanau attention: $\alpha_{t,s} = \text{softmax}(e_{t,s})$, direct precursor to transformers
- [ ] Perplexity = $\exp(\text{CE})$; trained LSTM achieves ~3-5 PPL on character-level tasks

## References

1. Hochreiter & Schmidhuber (1997). Long Short-Term Memory. *Neural Computation* 9(8).
2. Cho et al. (2014). Learning Phrase Representations using RNN Encoder-Decoder. *EMNLP*.
3. Bahdanau et al. (2015). Neural Machine Translation by Jointly Learning to Align. *ICLR*.
4. Pascanu et al. (2013). On the Difficulty of Training Recurrent Neural Networks. *ICML*.
5. Greff et al. (2017). LSTM: A Search Space Odyssey. *IEEE TNNLS*.
6. Gu & Dao (2024). Mamba: Linear-Time Sequence Modelling with Selective State Spaces. *ICLR*.